# Auditoria das features de fingerprinting segundo a taxonomia de Panda et al.

Este notebook **não treina modelos, não altera os Parquets originais e não recria fingerprints**.
Seu objetivo é interpretar as features de fingerprinting à luz da taxonomia de Panda, Malheiro e
Paiva (2020), verificando quais categorias musicais são cobertas pela representação e por que os
resultados tendem a favorecer `arousal` mais do que `valence`.

A tese examinada é simples:

> Os fingerprints capturam principalmente pistas acústicas ligadas a energia, saliência, brilho e
> altura local; por isso oferecem evidência mais direta para `arousal` do que para `valence`.

A análise principal prioriza medidas invariantes a ganho. Magnitudes e energias absolutas são
preservadas apenas como análise de sensibilidade. As categorias musicais inferidas a partir dos nomes
das colunas devem ser entendidas como **proxies acústicos**, não como transcrição musical completa.


## 2. Configurações e caminhos

Os quatro Parquets são entradas fixas e somente leitura. As saídas são organizadas por modo de
execução para que uma demonstração com uma única música nunca sobrescreva os resultados do corpus.


In [1]:
from pathlib import Path
import json
import math
import os
import re
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 180)
pio.templates.default = "plotly_white"
pio.renderers.default = "notebook_connected"

ANALISAR_APENAS_UMA_MUSICA = False
SONG_ID_ANALISE = 2
EXECUTAR_APENDICE_TECNICO = False

DEFAULT_DATA_DIR = Path(r"C:\dev\python\TCC Ajustado\Dados")
BASE_DIR = Path(os.environ.get("TCC_DADOS_DIR", DEFAULT_DATA_DIR))
DEFAULT_OUTPUT_ROOT = BASE_DIR.parent / "outputs" / "panda_taxonomia"
OUTPUT_ROOT = Path(os.environ.get("TCC_ANALISE_OUTPUT_DIR", DEFAULT_OUTPUT_ROOT))
RUN_NAME = f"musica_{SONG_ID_ANALISE:04d}" if ANALISAR_APENAS_UMA_MUSICA else "corpus"
OUTPUT_DIR = OUTPUT_ROOT / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    "rank": BASE_DIR / "fingerprint_rank" / "fingerprint_rank.parquet",
    "rank_raw": BASE_DIR / "fingerprint_rank" / "fingerprint_rank_raw_peaks.parquet",
    "band": BASE_DIR / "fingerprint_band_rank" / "fingerprint_band_rank.parquet",
    "band_raw": BASE_DIR / "fingerprint_band_rank" / "fingerprint_band_rank_raw_peaks.parquet",
}
missing = [f"{name}: {path}" for name, path in PATHS.items() if not path.exists()]
if missing:
    raise FileNotFoundError("Parquets obrigatórios não encontrados:\n" + "\n".join(missing))

INPUT_PARQUET_MTIMES_BEFORE = {name: path.stat().st_mtime_ns for name, path in PATHS.items()}

PANDA_CATEGORY_ORDER = [
    "Melodia",
    "Dinâmica",
    "Harmonia (proxy fraco)",
    "Timbre (proxy por banda)",
    "Ritmo (proxy fraco)",
    "Textura musical",
    "Forma musical",
    "Técnicas expressivas",
]
ABSENT_PANDA_CATEGORIES = ["Textura musical", "Forma musical", "Técnicas expressivas"]
BAND_ORDER = ["low", "mid", "high", "very_high"]
QUADRANT_ORDER = ["Q1", "Q2", "Q3", "Q4"]
QUADRANT_NAMES = {
    "Q1": "Alegre/Energético",
    "Q2": "Tenso/Raivoso",
    "Q3": "Triste/Melancólico",
    "Q4": "Calmo/Relaxado",
}
COLOR_PANDA = {
    "Melodia": "#2f6f9f",
    "Dinâmica": "#11a579",
    "Harmonia (proxy fraco)": "#7f3c8d",
    "Timbre (proxy por banda)": "#f58518",
    "Ritmo (proxy fraco)": "#e45756",
    "Textura musical": "#9d9d9d",
    "Forma musical": "#bdbdbd",
    "Técnicas expressivas": "#d0d0d0",
}
TARGET_COLORS = {"valence": "#7f3c8d", "arousal": "#11a579"}
TECHNIQUE_ORDER = ["Rank", "Band Rank", "Combinado"]

def improve_layout(fig, height=460, left=150, right=80, top=85, bottom=70):
    fig.update_layout(
        height=height,
        margin=dict(l=left, r=right, t=top, b=bottom),
        font=dict(size=13),
        title=dict(x=0.02, xanchor="left"),
    )
    fig.update_xaxes(showgrid=True, gridcolor="rgba(220,230,245,0.9)", zeroline=False)
    fig.update_yaxes(automargin=True)
    return fig

def save_plotly_figure(fig, stem):
    html_path = OUTPUT_DIR / f"{stem}.html"
    png_path = OUTPUT_DIR / f"{stem}.png"
    fig.write_html(html_path, include_plotlyjs="cdn", full_html=True)
    png_status = "ok"
    try:
        fig.write_image(png_path, scale=2)
    except Exception as exc:
        png_status = f"falhou: {type(exc).__name__}: {exc}"
        warnings.warn(f"PNG não exportado para {stem}: {png_status}")
    return {"html": str(html_path), "png": str(png_path), "png_status": png_status}

def empty_figure(title, message):
    fig = go.Figure()
    fig.add_annotation(
        x=0.5, y=0.5, xref="paper", yref="paper",
        text=message, showarrow=False, font=dict(size=15),
    )
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False)
    fig.update_layout(title=title)
    return improve_layout(fig, height=360, left=60, right=60)

print(f"Modo: {RUN_NAME}")
print(f"Dados: {BASE_DIR}")
print(f"Saídas: {OUTPUT_DIR}")
print(f"Apêndice técnico: {'ativo' if EXECUTAR_APENDICE_TECNICO else 'inativo'}")


Modo: corpus
Dados: C:\dev\python\TCC Ajustado\Dados
Saídas: C:\dev\python\TCC Ajustado\outputs\panda_taxonomia\corpus
Apêndice técnico: inativo


## 3. Carregamento somente leitura dos dados

No modo de música única, os quatro DataFrames são filtrados imediatamente. Todas as tabelas e figuras
subsequentes passam a ser ilustrativas daquela música e são gravadas em uma subpasta própria.


In [2]:
df_rank = pd.read_parquet(PATHS["rank"])
df_rank_raw = pd.read_parquet(PATHS["rank_raw"])
df_band = pd.read_parquet(PATHS["band"])
df_band_raw = pd.read_parquet(PATHS["band_raw"])

if ANALISAR_APENAS_UMA_MUSICA:
    available_ids = set(pd.to_numeric(df_rank["song_id"], errors="coerce").dropna().astype(int))
    if SONG_ID_ANALISE not in available_ids:
        raise ValueError(f"song_id={SONG_ID_ANALISE} não existe no corpus.")
    df_rank = df_rank[df_rank["song_id"].eq(SONG_ID_ANALISE)].copy()
    df_rank_raw = df_rank_raw[df_rank_raw["song_id"].eq(SONG_ID_ANALISE)].copy()
    df_band = df_band[df_band["song_id"].eq(SONG_ID_ANALISE)].copy()
    df_band_raw = df_band_raw[df_band_raw["song_id"].eq(SONG_ID_ANALISE)].copy()

shape_summary = pd.DataFrame([
    {"dataset": "fingerprint_rank", "linhas": len(df_rank), "colunas": df_rank.shape[1]},
    {"dataset": "fingerprint_rank_raw_peaks", "linhas": len(df_rank_raw), "colunas": df_rank_raw.shape[1]},
    {"dataset": "fingerprint_band_rank", "linhas": len(df_band), "colunas": df_band.shape[1]},
    {"dataset": "fingerprint_band_rank_raw_peaks", "linhas": len(df_band_raw), "colunas": df_band_raw.shape[1]},
])
display(shape_summary)

if ANALISAR_APENAS_UMA_MUSICA:
    warnings.warn(
        "Modo de música única: correlações e perfis taxonômicos têm baixa amostra e devem ser lidos "
        "somente como evidência ilustrativa."
    )


,dataset,linhas,colunas
0,fingerprint_rank,6506,222
1,fingerprint_rank_raw_peaks,195180,17
2,fingerprint_band_rank,6506,358
3,fingerprint_band_rank_raw_peaks,260240,27


## 4. Auditoria das colunas disponíveis

Identificadores, metadados e rótulos emocionais são excluídos do conjunto de features. A auditoria
também registra colunas constantes, que permanecem na cobertura taxonômica, mas recebem `N/A` nas
correlações por não possuírem variância válida.


In [3]:
LABEL_COLS = {"valence", "arousal", "quadrant", "quadrant_label"}
METADATA_COLS = {
    "song_id", "filename", "title", "artist", "genre", "source_file",
    "block_id", "block_start_sec", "block_end_sec", "block_duration_sec", "method",
}

def feature_columns(df):
    excluded = LABEL_COLS | METADATA_COLS
    return [
        col for col in df.columns
        if col not in excluded and pd.api.types.is_numeric_dtype(df[col])
    ]

feature_cols_rank = feature_columns(df_rank)
feature_cols_band = feature_columns(df_band)
leakage_check = sorted((LABEL_COLS | METADATA_COLS) & (set(feature_cols_rank) | set(feature_cols_band)))
assert not leakage_check, f"Metadados/rótulos detectados como features: {leakage_check}"

audit_columns = pd.DataFrame([
    {
        "dataset": "Rank",
        "features_numericas": len(feature_cols_rank),
        "features_constantes": sum(df_rank[c].nunique(dropna=True) <= 1 for c in feature_cols_rank),
        "linhas": len(df_rank),
        "musicas": df_rank["song_id"].nunique(),
    },
    {
        "dataset": "Band Rank",
        "features_numericas": len(feature_cols_band),
        "features_constantes": sum(df_band[c].nunique(dropna=True) <= 1 for c in feature_cols_band),
        "linhas": len(df_band),
        "musicas": df_band["song_id"].nunique(),
    },
])
display(audit_columns)


,dataset,features_numericas,features_constantes,linhas,musicas
0,Rank,207,0,6506,1802
1,Band Rank,347,45,6506,1802


## 5. Dicionário de features de fingerprinting

O mapeamento é deliberadamente conservador:

- energia, magnitude e score representam **Dinâmica**;
- frequência, MIDI e oitava representam **altura local**, não contorno melódico;
- classe de pitch e semitom são **proxies harmônicos fracos**, não acordes ou tonalidade;
- contagens e intervalos entre eventos são **proxies rítmicos fracos**;
- ranks por banda representam organização espectral, mas ranks constantes não sustentam associação;
- textura, forma e técnicas expressivas permanecem ausentes.

A representação concentra-se, portanto, em saliências espectrais locais: energia, magnitude,
distribuição por bandas, frequência, oitava e classe aproximada de pitch. Isso não equivale a
harmonia funcional, forma, textura, expressividade ou melodia completa.


In [4]:
def infer_band(col):
    c = col.lower()
    if "very_high" in c:
        return "very_high"
    for band in ["low", "mid", "high", "total"]:
        if re.search(rf"(^|_)({band})(_|$)", c):
            return band
    return "geral"

def infer_feature_type(col):
    c = col.lower()
    if "energy" in c:
        return "energia"
    if "score" in c:
        return "saliência normalizada"
    if "magnitude" in c or re.search(r"(^|_)mag(_|$)", c):
        return "magnitude"
    if "pitch_class" in c:
        return "classe de pitch"
    if "semitone" in c:
        return "semitom"
    if "midi" in c:
        return "MIDI"
    if "octave" in c:
        return "oitava"
    if "freq" in c:
        return "frequência"
    if "event_count" in c or c.startswith("n_") or "peak_gap" in c:
        return "densidade de eventos"
    if "rank" in c:
        return "ordem de saliência"
    return "outro"

def classify_panda_category(col):
    c = col.lower()
    if "energy" in c or "score" in c or "magnitude" in c or re.search(r"(^|_)mag(_|$)", c):
        return "Dinâmica"
    if "pitch_class" in c or "semitone" in c or "unique_pitch_classes" in c or "octave_duplicate" in c:
        return "Harmonia (proxy fraco)"
    if "event_count" in c or c.startswith("n_") or "peak_gap" in c:
        return "Ritmo (proxy fraco)"
    if "freq" in c or "midi" in c or "octave" in c:
        return "Melodia"
    if "rank" in c and infer_band(c) != "geral":
        return "Timbre (proxy por banda)"
    return "Não classificado"

def coverage_level(category):
    return {
        "Dinâmica": "cobertura direta por energia/saliência",
        "Melodia": "proxy de altura local",
        "Harmonia (proxy fraco)": "proxy fraco de classe de altura",
        "Ritmo (proxy fraco)": "proxy fraco de densidade temporal",
        "Timbre (proxy por banda)": "proxy espectral por banda",
        "Textura musical": "ausente",
        "Forma musical": "ausente",
        "Técnicas expressivas": "ausente",
    }.get(category, "não classificado")

def category_observation(category):
    return {
        "Dinâmica": "energia, magnitude e saliência; associação direta com ativação",
        "Melodia": "altura local; não mede contorno, fraseado ou motivo",
        "Harmonia (proxy fraco)": "classe de altura; não mede acordes, progressões, modo ou tonalidade",
        "Ritmo (proxy fraco)": "densidade/espaçamento; não mede tempo, métrica, groove ou síncope",
        "Timbre (proxy por banda)": "organização espectral por faixa; ranks disponíveis são constantes",
        "Textura musical": "sem descritor de camadas ou separação de fontes",
        "Forma musical": "sem descritor de seções, recorrência ou macroforma",
        "Técnicas expressivas": "sem vibrato, articulação ou parâmetros de performance",
    }[category]

def build_feature_dictionary(columns, origin, df):
    rows = []
    for col in columns:
        category = classify_panda_category(col)
        normalized_dynamic = (
            category == "Dinâmica"
            and ("norm" in col.lower() or col.lower().startswith("score_"))
        )
        gain_invariant = category != "Dinâmica" or normalized_dynamic
        rows.append({
            "coluna": col,
            "origem": origin,
            "tipo_feature": infer_feature_type(col),
            "banda": infer_band(col),
            "categoria_panda": category,
            "nivel_cobertura": coverage_level(category),
            "invariante_a_ganho": gain_invariant,
            "escopo_evidencia": (
                "principal: invariante a ganho"
                if gain_invariant else "sensibilidade: nível absoluto"
            ),
            "variancia_valida": df[col].nunique(dropna=True) > 1,
            "observacao": category_observation(category),
        })
    return pd.DataFrame(rows)

df_features = pd.concat([
    build_feature_dictionary(feature_cols_rank, "Rank", df_rank),
    build_feature_dictionary(feature_cols_band, "Band Rank", df_band),
], ignore_index=True)

nao_classificadas = df_features[df_features["categoria_panda"].eq("Não classificado")]
invariant_count = int(df_features["invariante_a_ganho"].sum())
absolute_dynamic_count = int((~df_features["invariante_a_ganho"]).sum())

print(f"Features mapeadas: {len(df_features)}")
print(f"Features invariantes a ganho: {invariant_count}")
print(f"Features absolutas mantidas só na sensibilidade: {absolute_dynamic_count}")
print(f"Features não classificadas: {len(nao_classificadas)}")

df_features.to_csv(
    OUTPUT_DIR / "00_dicionario_features_fingerprint.csv",
    index=False,
    encoding="utf-8-sig",
)
display(df_features.head(12))


Features mapeadas: 554
Features invariantes a ganho: 467
Features absolutas mantidas só na sensibilidade: 87
Features não classificadas: 0


,coluna,origem,tipo_feature,banda,categoria_panda,nivel_cobertura,invariante_a_ganho,escopo_evidencia,variancia_valida,observacao
0,fp_top_10_freq_hz,Rank,frequência,geral,Melodia,proxy de altura local,True,principal: invariante a ganho,True,"altura local; não mede contorno, fraseado ou motivo"
1,fp_top_10_magnitude_db,Rank,magnitude,geral,Dinâmica,cobertura direta por energia/saliência,False,sensibilidade: nível absoluto,True,"energia, magnitude e saliência; associação direta com ativação"
2,fp_top_10_magnitude_norm_block,Rank,magnitude,geral,Dinâmica,cobertura direta por energia/saliência,True,principal: invariante a ganho,True,"energia, magnitude e saliência; associação direta com ativação"
3,fp_top_10_midi,Rank,MIDI,geral,Melodia,proxy de altura local,True,principal: invariante a ganho,True,"altura local; não mede contorno, fraseado ou motivo"
4,fp_top_10_octave,Rank,oitava,geral,Melodia,proxy de altura local,True,principal: invariante a ganho,True,"altura local; não mede contorno, fraseado ou motivo"
5,fp_top_10_semitone_approx,Rank,semitom,geral,Harmonia (proxy fraco),proxy fraco de classe de altura,True,principal: invariante a ganho,True,"classe de altura; não mede acordes, progressões, modo ou tonalidade"
6,fp_top_11_freq_hz,Rank,frequência,geral,Melodia,proxy de altura local,True,principal: invariante a ganho,True,"altura local; não mede contorno, fraseado ou motivo"
7,fp_top_11_magnitude_db,Rank,magnitude,geral,Dinâmica,cobertura direta por energia/saliência,False,sensibilidade: nível absoluto,True,"energia, magnitude e saliência; associação direta com ativação"
8,fp_top_11_magnitude_norm_block,Rank,magnitude,geral,Dinâmica,cobertura direta por energia/saliência,True,principal: invariante a ganho,True,"energia, magnitude e saliência; associação direta com ativação"
9,fp_top_11_midi,Rank,MIDI,geral,Melodia,proxy de altura local,True,principal: invariante a ganho,True,"altura local; não mede contorno, fraseado ou motivo"


## 6. Mapeamento das features para Panda et al.

A contagem taxonômica descreve o que existe no vetor, não a quantidade de informação independente.
Por exemplo, as 40 colunas de rank por banda são contabilizadas como cobertura espectral, mas são
constantes no corpus e por isso não recebem correlação emocional válida.


In [5]:
mapping_audit = (
    df_features
    .groupby(
        ["origem", "categoria_panda", "nivel_cobertura", "escopo_evidencia"],
        as_index=False,
    )
    .agg(
        quantidade_features=("coluna", "count"),
        features_com_variancia=("variancia_valida", "sum"),
    )
)
mapping_audit["categoria_panda"] = pd.Categorical(
    mapping_audit["categoria_panda"], categories=PANDA_CATEGORY_ORDER, ordered=True
)
mapping_audit = mapping_audit.sort_values(
    ["origem", "categoria_panda", "escopo_evidencia"]
)
display(mapping_audit)


,origem,categoria_panda,nivel_cobertura,escopo_evidencia,quantidade_features,features_com_variancia
3,Band Rank,Melodia,proxy de altura local,principal: invariante a ganho,146,146
0,Band Rank,Dinâmica,cobertura direta por energia/saliência,principal: invariante a ganho,58,58
1,Band Rank,Dinâmica,cobertura direta por energia/saliência,sensibilidade: nível absoluto,54,54
2,Band Rank,Harmonia (proxy fraco),proxy fraco de classe de altura,principal: invariante a ganho,44,44
5,Band Rank,Timbre (proxy por banda),proxy espectral por banda,principal: invariante a ganho,40,0
4,Band Rank,Ritmo (proxy fraco),proxy fraco de densidade temporal,principal: invariante a ganho,5,0
9,Rank,Melodia,proxy de altura local,principal: invariante a ganho,101,101
6,Rank,Dinâmica,cobertura direta por energia/saliência,principal: invariante a ganho,31,31
7,Rank,Dinâmica,cobertura direta por energia/saliência,sensibilidade: nível absoluto,33,33
8,Rank,Harmonia (proxy fraco),proxy fraco de classe de altura,principal: invariante a ganho,37,37


## 7. Cobertura das categorias musicais

A cobertura inclui explicitamente as categorias ausentes e separa as duas técnicas: **Rank**
(207 features) e **Band Rank** (347 features). O percentual é calculado dentro de cada técnica; a
visão `Combinado` preserva o total de 554 features apenas como síntese. A coluna de variância válida
impede que mera presença nominal seja confundida com evidência estatística.


In [6]:
coverage_rows = []
technique_subsets = {
    "Rank": df_features[df_features["origem"].eq("Rank")],
    "Band Rank": df_features[df_features["origem"].eq("Band Rank")],
    "Combinado": df_features,
}
for technique, technique_features in technique_subsets.items():
    total_technique = len(technique_features)
    for category in PANDA_CATEGORY_ORDER:
        subset = technique_features[technique_features["categoria_panda"].eq(category)]
        coverage_rows.append({
            "tecnica": technique,
            "categoria_panda": category,
            "quantidade_features": len(subset),
            "features_com_variancia": int(subset["variancia_valida"].sum()) if len(subset) else 0,
            "percentual": (len(subset) / total_technique * 100) if total_technique else 0,
            "nivel_cobertura": (
                subset["nivel_cobertura"].iloc[0] if len(subset) else coverage_level(category)
            ),
            "interpretacao": category_observation(category),
        })
coverage = pd.DataFrame(coverage_rows)
coverage["tecnica"] = pd.Categorical(
    coverage["tecnica"], categories=TECHNIQUE_ORDER, ordered=True
)
coverage["categoria_panda"] = pd.Categorical(
    coverage["categoria_panda"], categories=PANDA_CATEGORY_ORDER, ordered=True
)
coverage = coverage.sort_values(["tecnica", "categoria_panda"]).reset_index(drop=True)

tabela_cobertura = coverage.rename(columns={
    "tecnica": "Técnica de fingerprint",
    "categoria_panda": "Categoria de Panda",
    "quantidade_features": "Quantidade de features",
    "features_com_variancia": "Features com variância válida",
    "percentual": "Percentual",
    "nivel_cobertura": "Nível de cobertura",
    "interpretacao": "Interpretação",
})
tabela_cobertura.to_csv(
    OUTPUT_DIR / "01_tabela_cobertura_panda.csv", index=False, encoding="utf-8-sig"
)
tabela_cobertura.to_latex(
    OUTPUT_DIR / "01_tabela_cobertura_panda.tex",
    index=False, escape=False, float_format="%.2f",
)
for technique, stem in [("Rank", "rank"), ("Band Rank", "band_rank")]:
    technique_table = tabela_cobertura[
        tabela_cobertura["Técnica de fingerprint"].astype(str).eq(technique)
    ]
    technique_table.to_csv(
        OUTPUT_DIR / f"01_tabela_cobertura_{stem}.csv",
        index=False, encoding="utf-8-sig",
    )
    technique_table.to_latex(
        OUTPUT_DIR / f"01_tabela_cobertura_{stem}.tex",
        index=False, escape=False, float_format="%.2f",
    )
display(tabela_cobertura.round({"Percentual": 2}))


,Técnica de fingerprint,Categoria de Panda,Quantidade de features,Features com variância válida,Percentual,Nível de cobertura,Interpretação
0,Rank,Melodia,101,101,48.79,proxy de altura local,"altura local; não mede contorno, fraseado ou motivo"
1,Rank,Dinâmica,64,64,30.92,cobertura direta por energia/saliência,"energia, magnitude e saliência; associação direta com ativação"
2,Rank,Harmonia (proxy fraco),37,37,17.87,proxy fraco de classe de altura,"classe de altura; não mede acordes, progressões, modo ou tonalidade"
3,Rank,Timbre (proxy por banda),0,0,0.00,proxy espectral por banda,organização espectral por faixa; ranks disponíveis são constantes
4,Rank,Ritmo (proxy fraco),5,5,2.42,proxy fraco de densidade temporal,"densidade/espaçamento; não mede tempo, métrica, groove ou síncope"
5,Rank,Textura musical,0,0,0.00,ausente,sem descritor de camadas ou separação de fontes
6,Rank,Forma musical,0,0,0.00,ausente,"sem descritor de seções, recorrência ou macroforma"
7,Rank,Técnicas expressivas,0,0,0.00,ausente,"sem vibrato, articulação ou parâmetros de performance"
8,Band Rank,Melodia,146,146,42.07,proxy de altura local,"altura local; não mede contorno, fraseado ou motivo"
9,Band Rank,Dinâmica,112,112,32.28,cobertura direta por energia/saliência,"energia, magnitude e saliência; associação direta com ativação"


In [7]:
plot_coverage = coverage[
    coverage["tecnica"].isin(["Rank", "Band Rank"])
].sort_values(["tecnica", "categoria_panda"], ascending=[True, False]).copy()
coverage_order = (
    coverage[coverage["tecnica"].astype(str).eq("Combinado")]
    .sort_values(
        ["quantidade_features", "categoria_panda"],
        ascending=[False, True],
    )["categoria_panda"]
    .astype(str)
    .tolist()
)
fig_1 = px.bar(
    plot_coverage,
    x="quantidade_features",
    y="categoria_panda",
    orientation="h",
    color="categoria_panda",
    facet_col="tecnica",
    category_orders={
        "tecnica": ["Rank", "Band Rank"],
        "categoria_panda": coverage_order,
    },
    facet_col_spacing=0.10,
    color_discrete_map=COLOR_PANDA,
    text=plot_coverage.apply(
        lambda r: f'{int(r["quantidade_features"])} ({r["percentual"]:.1f}%)', axis=1
    ),
    title="Figura 1 — Cobertura taxonômica separada por técnica de fingerprint",
    labels={
        "quantidade_features": "Quantidade de features",
        "categoria_panda": "",
        "tecnica": "Técnica",
    },
)
fig_1.update_traces(textposition="outside", cliponaxis=False)
fig_1.update_layout(showlegend=False)
fig_1.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig_1.update_xaxes(range=[0, 180], title_text="")
fig_1.update_yaxes(
    categoryorder="array",
    categoryarray=coverage_order,
    autorange="reversed",
)
fig_1.add_annotation(
    x=0.5, y=-0.10, xref="paper", yref="paper",
    text="Quantidade de features", showarrow=False, font=dict(size=15),
)
fig_1.update_layout(width=1100)
improve_layout(fig_1, height=560, left=220, right=100, bottom=95)
save_plotly_figure(fig_1, "fig_01_cobertura_panda")
fig_1.show()


## 8. Associação com valence e arousal

A evidência principal contém 467 features invariantes a ganho. Dentro de Dinâmica, apenas
`energy_norm_*`, `score_*` e magnitudes `_norm` são usadas; altura, classe de pitch e contagens já são
invariantes por construção. As 87 magnitudes e energias absolutas entram somente na análise de
sensibilidade.

As correlações são calculadas em dois níveis:

1. por bloco, preservando a resolução temporal do TCC;
2. por música, após média de features e targets dentro de cada `song_id`.

Cada resultado é apresentado separadamente para **Rank** e **Band Rank**, além de uma linha combinada
usada somente como síntese geral.


In [8]:
def safe_abs_corr(x, y):
    x = pd.to_numeric(x, errors="coerce").replace([np.inf, -np.inf], np.nan)
    y = pd.to_numeric(y, errors="coerce").replace([np.inf, -np.inf], np.nan)
    valid = x.notna() & y.notna()
    if valid.sum() < 3 or x[valid].nunique() <= 1 or y[valid].nunique() <= 1:
        return np.nan
    return abs(float(x[valid].corr(y[valid])))

def correlation_rows(df, feature_meta, origin, scope_name, level):
    selected = feature_meta[feature_meta["origem"].eq(origin)].copy()
    if scope_name == "Principal (invariante a ganho)":
        selected = selected[selected["invariante_a_ganho"]]
    feature_names = [c for c in selected["coluna"] if c in df.columns]
    analysis_df = df
    if level == "Música":
        analysis_df = (
            df.groupby("song_id", as_index=False)[feature_names + ["valence", "arousal"]]
            .mean(numeric_only=True)
        )
    rows = []
    for row in selected.itertuples(index=False):
        for target in ["valence", "arousal"]:
            rows.append({
                "origem": origin,
                "feature": row.coluna,
                "categoria_panda": row.categoria_panda,
                "escopo": scope_name,
                "nivel_analise": level,
                "target": target,
                "corr_abs": safe_abs_corr(analysis_df[row.coluna], analysis_df[target]),
            })
    return rows

corr_rows = []
for scope_name in ["Principal (invariante a ganho)", "Sensibilidade (todas as features)"]:
    for level in ["Bloco", "Música"]:
        corr_rows.extend(correlation_rows(df_rank, df_features, "Rank", scope_name, level))
        corr_rows.extend(correlation_rows(df_band, df_features, "Band Rank", scope_name, level))
feature_correlations = pd.DataFrame(corr_rows)

aggregate_rows = []
for scope_name in ["Principal (invariante a ganho)", "Sensibilidade (todas as features)"]:
    for level in ["Bloco", "Música"]:
        for technique in TECHNIQUE_ORDER:
            for category in PANDA_CATEGORY_ORDER:
                for target in ["valence", "arousal"]:
                    subset = feature_correlations[
                        feature_correlations["escopo"].eq(scope_name)
                        & feature_correlations["nivel_analise"].eq(level)
                        & feature_correlations["categoria_panda"].eq(category)
                        & feature_correlations["target"].eq(target)
                    ]
                    if technique != "Combinado":
                        subset = subset[subset["origem"].eq(technique)]
                    valid = subset["corr_abs"].dropna()
                    aggregate_rows.append({
                        "tecnica": technique,
                        "categoria_panda": category,
                        "escopo": scope_name,
                        "nivel_analise": level,
                        "target": target,
                        "corr_abs_media": valid.mean() if len(valid) else np.nan,
                        "corr_abs_mediana": valid.median() if len(valid) else np.nan,
                        "corr_abs_maxima": valid.max() if len(valid) else np.nan,
                        "features_disponiveis": subset["feature"].nunique(),
                        "features_validas": subset.loc[
                            subset["corr_abs"].notna(), "feature"
                        ].nunique(),
                    })
category_correlations = pd.DataFrame(aggregate_rows)

tabela_correlacoes = category_correlations.rename(columns={
    "tecnica": "Técnica de fingerprint",
    "categoria_panda": "Categoria",
    "escopo": "Escopo",
    "nivel_analise": "Nível de análise",
    "target": "Target",
    "corr_abs_media": "Correlação média |r|",
    "corr_abs_mediana": "Correlação mediana |r|",
    "corr_abs_maxima": "Correlação máxima |r|",
    "features_disponiveis": "Features disponíveis",
    "features_validas": "Features válidas",
})
tabela_correlacoes.to_csv(
    OUTPUT_DIR / "02_tabela_correlacao_categoria.csv", index=False, encoding="utf-8-sig"
)
tabela_correlacoes.to_latex(
    OUTPUT_DIR / "02_tabela_correlacao_categoria.tex",
    index=False, escape=False, na_rep="N/A", float_format="%.4f",
)
for technique, stem in [("Rank", "rank"), ("Band Rank", "band_rank")]:
    technique_table = tabela_correlacoes[
        tabela_correlacoes["Técnica de fingerprint"].eq(technique)
    ]
    technique_table.to_csv(
        OUTPUT_DIR / f"02_tabela_correlacao_{stem}.csv",
        index=False, encoding="utf-8-sig",
    )
    technique_table.to_latex(
        OUTPUT_DIR / f"02_tabela_correlacao_{stem}.tex",
        index=False, escape=False, na_rep="N/A", float_format="%.4f",
    )

principal_block = category_correlations[
    category_correlations["escopo"].eq("Principal (invariante a ganho)")
    & category_correlations["nivel_analise"].eq("Bloco")
]
display(
    tabela_correlacoes[
        tabela_correlacoes["Escopo"].eq("Principal (invariante a ganho)")
        & tabela_correlacoes["Nível de análise"].eq("Bloco")
    ].round(4)
)


,Técnica de fingerprint,Categoria,Escopo,Nível de análise,Target,Correlação média |r|,Correlação mediana |r|,Correlação máxima |r|,Features disponíveis,Features válidas
0,Rank,Melodia,Principal (invariante a ganho),Bloco,valence,0.0384,0.0159,0.1777,101,101
1,Rank,Melodia,Principal (invariante a ganho),Bloco,arousal,0.0673,0.0274,0.3536,101,101
2,Rank,Dinâmica,Principal (invariante a ganho),Bloco,valence,0.0627,0.0551,0.2773,31,31
3,Rank,Dinâmica,Principal (invariante a ganho),Bloco,arousal,0.2485,0.2337,0.6396,31,31
4,Rank,Harmonia (proxy fraco),Principal (invariante a ganho),Bloco,valence,0.0394,0.0106,0.1876,37,37
5,Rank,Harmonia (proxy fraco),Principal (invariante a ganho),Bloco,arousal,0.0670,0.0261,0.3656,37,37
6,Rank,Timbre (proxy por banda),Principal (invariante a ganho),Bloco,valence,NaN,NaN,NaN,0,0
7,Rank,Timbre (proxy por banda),Principal (invariante a ganho),Bloco,arousal,NaN,NaN,NaN,0,0
8,Rank,Ritmo (proxy fraco),Principal (invariante a ganho),Bloco,valence,0.0696,0.0753,0.1699,5,5
9,Rank,Ritmo (proxy fraco),Principal (invariante a ganho),Bloco,arousal,0.0855,0.0509,0.1490,5,5


In [9]:
corr_plot = principal_block[principal_block["corr_abs_media"].notna()].copy()
corr_plot = corr_plot[corr_plot["tecnica"].isin(["Rank", "Band Rank"])]
correlation_order = (
    principal_block[
        principal_block["tecnica"].eq("Combinado")
        & principal_block["target"].eq("arousal")
        & principal_block["corr_abs_media"].notna()
    ]
    .sort_values("corr_abs_media", ascending=False)["categoria_panda"]
    .tolist()
)
corr_plot["categoria_panda"] = pd.Categorical(
    corr_plot["categoria_panda"], categories=PANDA_CATEGORY_ORDER, ordered=True
)
corr_plot = corr_plot.sort_values(
    ["tecnica", "categoria_panda", "target"], ascending=[True, False, True]
)
corr_plot["rotulo"] = corr_plot.apply(
    lambda r: f'{r["corr_abs_media"]:.3f} | n={int(r["features_validas"])}', axis=1
)
fig_2 = px.bar(
    corr_plot,
    x="corr_abs_media",
    y="categoria_panda",
    color="target",
    orientation="h",
    barmode="group",
    facet_col="tecnica",
    category_orders={
        "tecnica": ["Rank", "Band Rank"],
        "categoria_panda": correlation_order,
    },
    facet_col_spacing=0.10,
    text="rotulo",
    color_discrete_map=TARGET_COLORS,
    title="Figura 2 — Associação emocional separada por técnica de fingerprint",
    labels={
        "corr_abs_media": "Correlação absoluta média",
        "categoria_panda": "",
        "target": "Target",
        "tecnica": "Técnica",
    },
)
fig_2.update_traces(textposition="outside", cliponaxis=False)
fig_2.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig_2.update_xaxes(range=[0, 0.37], title_text="")
fig_2.update_yaxes(
    categoryorder="array",
    categoryarray=correlation_order,
    autorange="reversed",
)
fig_2.add_annotation(
    x=0.5, y=-0.10, xref="paper", yref="paper",
    text="Correlação absoluta média", showarrow=False, font=dict(size=15),
)
fig_2.update_layout(width=1150)
improve_layout(fig_2, height=560, left=220, right=150, bottom=95)
save_plotly_figure(fig_2, "fig_02_correlacao_categoria")
fig_2.show()


## 9. Diferença arousal versus valence

A tabela abaixo transforma as duas correlações médias em uma comparação direta. Gap positivo indica
maior associação com `arousal`. Categorias sem features ou sem variância válida permanecem visíveis
como `N/A`.


In [10]:
gap_index = pd.MultiIndex.from_product(
    [TECHNIQUE_ORDER, PANDA_CATEGORY_ORDER],
    names=["tecnica", "categoria_panda"],
)
gap_base = (
    principal_block.pivot_table(
        index=["tecnica", "categoria_panda"],
        columns="target",
        values="corr_abs_media",
        aggfunc="first",
    )
    .reindex(gap_index)
    .reset_index()
)
gap_base["diferenca_arousal_valence"] = gap_base["arousal"] - gap_base["valence"]
gap_base["interpretacao"] = np.select(
    [
        gap_base["diferenca_arousal_valence"].isna(),
        gap_base["diferenca_arousal_valence"] > 0.02,
        gap_base["diferenca_arousal_valence"] < -0.02,
    ],
    [
        "N/A: categoria ausente ou sem variância válida",
        "associação mais forte com arousal",
        "associação mais forte com valence",
    ],
    default="associação semelhante entre os eixos",
)
tabela_gap = gap_base.rename(columns={
    "tecnica": "Técnica de fingerprint",
    "categoria_panda": "Categoria",
    "arousal": "|r| médio arousal",
    "valence": "|r| médio valence",
    "diferenca_arousal_valence": "Diferença",
    "interpretacao": "Interpretação",
})
tabela_gap.to_csv(
    OUTPUT_DIR / "03_tabela_diferenca_arousal_valence.csv", index=False, encoding="utf-8-sig"
)
tabela_gap.to_latex(
    OUTPUT_DIR / "03_tabela_diferenca_arousal_valence.tex",
    index=False, escape=False, na_rep="N/A", float_format="%.4f",
)
for technique, stem in [("Rank", "rank"), ("Band Rank", "band_rank")]:
    technique_table = tabela_gap[
        tabela_gap["Técnica de fingerprint"].eq(technique)
    ]
    technique_table.to_csv(
        OUTPUT_DIR / f"03_tabela_diferenca_{stem}.csv",
        index=False, encoding="utf-8-sig",
    )
    technique_table.to_latex(
        OUTPUT_DIR / f"03_tabela_diferenca_{stem}.tex",
        index=False, escape=False, na_rep="N/A", float_format="%.4f",
    )
display(tabela_gap.round(4))

normalized_dynamic = feature_correlations[
    feature_correlations["escopo"].eq("Principal (invariante a ganho)")
    & feature_correlations["nivel_analise"].eq("Bloco")
    & feature_correlations["categoria_panda"].eq("Dinâmica")
]
feature_ratio = (
    normalized_dynamic.pivot_table(
        index=["origem", "feature"], columns="target", values="corr_abs", aggfunc="first"
    )
    .reset_index()
    .dropna(subset=["arousal", "valence"])
)
feature_ratio["razao_arousal_valence"] = feature_ratio["arousal"] / (
    feature_ratio["valence"] + 1e-12
)
feature_ratio["gap"] = feature_ratio["arousal"] - feature_ratio["valence"]
ratio_plot = feature_ratio.nlargest(18, "gap").sort_values(
    "razao_arousal_valence", ascending=False
)

if ratio_plot.empty:
    fig_3 = empty_figure(
        "Figura 3 — Razão arousal/valence dos descritores normalizados",
        "Correlação indisponível neste modo de execução.",
    )
else:
    fig_3 = px.bar(
        ratio_plot,
        x="razao_arousal_valence",
        y="feature",
        orientation="h",
        color="origem",
        title="Figura 3 — Razão arousal/valence dos descritores normalizados",
        labels={
            "razao_arousal_valence": "|r arousal| / |r valence|",
            "feature": "Descritor",
            "origem": "Origem",
        },
    )
    fig_3.add_vline(x=1, line_dash="dash", line_color="#333")
    fig_3.update_yaxes(
        categoryorder="array",
        categoryarray=ratio_plot["feature"].tolist(),
        autorange="reversed",
    )
    improve_layout(fig_3, height=650, left=290, right=80)
save_plotly_figure(fig_3, "fig_03_razao_arousal_valence")
fig_3.show()


target,Técnica de fingerprint,Categoria,|r| médio arousal,|r| médio valence,Diferença,Interpretação
0,Rank,Melodia,0.0673,0.0384,0.0289,associação mais forte com arousal
1,Rank,Dinâmica,0.2485,0.0627,0.1858,associação mais forte com arousal
2,Rank,Harmonia (proxy fraco),0.0670,0.0394,0.0276,associação mais forte com arousal
3,Rank,Timbre (proxy por banda),NaN,NaN,NaN,N/A: categoria ausente ou sem variância válida
4,Rank,Ritmo (proxy fraco),0.0855,0.0696,0.0160,associação semelhante entre os eixos
5,Rank,Textura musical,NaN,NaN,NaN,N/A: categoria ausente ou sem variância válida
6,Rank,Forma musical,NaN,NaN,NaN,N/A: categoria ausente ou sem variância válida
7,Rank,Técnicas expressivas,NaN,NaN,NaN,N/A: categoria ausente ou sem variância válida
8,Band Rank,Melodia,0.0644,0.0334,0.0310,associação mais forte com arousal
9,Band Rank,Dinâmica,0.2980,0.0769,0.2211,associação mais forte com arousal


## 10. Perfil por quadrante de Russell

Os quadrantes permitem visualizar a dificuldade central: Q1 e Q2 compartilham alto `arousal`, mas
possuem sinais opostos de `valence`. A análise usa apenas energia, saliência e magnitudes normalizadas.
Como esses descritores por banda pertencem à técnica **Fingerprint Band Rank**, esta seção é
explicitamente exclusiva dela. O heatmap padroniza cada descritor entre os quadrantes para comparar
perfis, não amplitudes absolutas.


In [11]:
def quadrant_code(df):
    if "quadrant" in df.columns and df["quadrant"].notna().any():
        extracted = df["quadrant"].astype(str).str.extract(r"(Q[1-4])")[0]
        if extracted.notna().any():
            return extracted
    return df["quadrant_label"].astype(str).str.extract(r"(Q[1-4])")[0]

band_q = df_band.copy()
band_q["Q"] = quadrant_code(band_q)
energy_norm = [c for c in [f"energy_norm_{b}" for b in BAND_ORDER] if c in band_q.columns]
score_norm = [c for c in [f"score_{b}" for b in BAND_ORDER] if c in band_q.columns]
mag_norm = [
    c for c in [
        "fp_total_mag_mean_norm_block",
        "fp_low_mag_mean_norm_block",
        "fp_mid_mag_mean_norm_block",
        "fp_high_mag_mean_norm_block",
        "fp_very_high_mag_mean_norm_block",
    ] if c in band_q.columns
]
profile_features = energy_norm + score_norm + mag_norm

quadrant_numeric = (
    band_q.dropna(subset=["Q"])
    .groupby("Q")
    .agg(
        n_blocos=("Q", "size"),
        valence_medio=("valence", "mean"),
        arousal_medio=("arousal", "mean"),
        **{feature: (feature, "mean") for feature in profile_features},
    )
    .reindex(QUADRANT_ORDER)
)

profile_matrix = quadrant_numeric[profile_features].T
row_std = profile_matrix.std(axis=1, ddof=0).replace(0, np.nan)
profile_z = profile_matrix.sub(profile_matrix.mean(axis=1), axis=0).div(row_std, axis=0)

interpretation_by_q = {
    "Q1": "alta ativação e valência positiva; energia não basta para separá-lo de Q2",
    "Q2": "alta ativação e valência negativa; exige pistas de dissonância/modo ausentes",
    "Q3": "baixa ativação e valência negativa",
    "Q4": "baixa ativação e valência positiva; diferenciação depende de pistas de alto nível",
}
table_rows = []
for q in QUADRANT_ORDER:
    z_values = profile_z[q].dropna() if q in profile_z else pd.Series(dtype=float)
    if not z_values.empty:
        top_descriptors = ", ".join(z_values.nlargest(3).index)
    else:
        raw_values = profile_matrix[q].dropna() if q in profile_matrix else pd.Series(dtype=float)
        top_descriptors = ", ".join(raw_values.nlargest(3).index) if not raw_values.empty else "N/A"
    table_rows.append({
        "Quadrante": q,
        "n_blocos": quadrant_numeric.loc[q, "n_blocos"],
        "valence médio": quadrant_numeric.loc[q, "valence_medio"],
        "arousal médio": quadrant_numeric.loc[q, "arousal_medio"],
        "principais descritores normalizados": top_descriptors,
        "interpretação": interpretation_by_q[q],
    })
tabela_quadrantes = pd.DataFrame(table_rows)
tabela_quadrantes.to_csv(
    OUTPUT_DIR / "04_tabela_perfil_quadrante.csv", index=False, encoding="utf-8-sig"
)
tabela_quadrantes.to_latex(
    OUTPUT_DIR / "04_tabela_perfil_quadrante.tex",
    index=False, escape=False, na_rep="N/A", float_format="%.4f",
)
display(tabela_quadrantes.round(4))

if profile_z.notna().any().any():
    fig_4 = px.imshow(
        profile_z,
        aspect="auto",
        color_continuous_scale="RdBu_r",
        color_continuous_midpoint=0,
        text_auto=".2f",
        title="Figura 4 — Perfil padronizado dos descritores normalizados por quadrante",
        labels={"x": "Quadrante", "y": "Descritor normalizado", "color": "z-score"},
    )
    improve_layout(fig_4, height=620, left=280, right=110)
else:
    fig_4 = empty_figure(
        "Figura 4 — Perfil padronizado por quadrante",
        "São necessários ao menos dois quadrantes com dados para padronizar o perfil.",
    )
save_plotly_figure(fig_4, "fig_04_heatmap_quadrantes")
fig_4.show()


,Quadrante,n_blocos,valence médio,arousal médio,principais descritores normalizados,interpretação
0,Q1,3217,0.2585,0.3011,"score_low, score_very_high, energy_norm_very_high",alta ativação e valência positiva; energia não basta para separá-lo de Q2
1,Q2,1019,-0.2077,0.2693,"fp_mid_mag_mean_norm_block, energy_norm_mid, fp_total_mag_mean_norm_block",alta ativação e valência negativa; exige pistas de dissonância/modo ausentes
2,Q3,1382,-0.1771,-0.2159,"score_mid, fp_mid_mag_mean_norm_block, fp_high_mag_mean_norm_block",baixa ativação e valência negativa
3,Q4,888,0.1420,-0.1314,"score_low, score_mid, fp_mid_mag_mean_norm_block",baixa ativação e valência positiva; diferenciação depende de pistas de alto nível


## 11. Figuras finais e mapa de lacunas

O mapa de lacunas resume a função argumentativa das análises: quais conceitos são observados,
que tipo de evidência existe, quais limitações permanecem e qual impacto é esperado em cada eixo
emocional.


In [12]:
gap_map = pd.DataFrame([
    ["Melodia", "101 features: altura local", "146 features: altura local por banda", "sem contorno ou fraseado", "baixo/moderado em ambos"],
    ["Dinâmica", "64 features globais", "112 features globais e por banda", "versões absolutas sofrem efeito de ganho", "forte em arousal"],
    ["Harmonia", "37 proxies de classe de pitch", "44 proxies de classe de pitch por banda", "sem acordes, modo ou progressões", "limita valence"],
    ["Timbre", "ausente como categoria dedicada", "40 ranks por banda, todos constantes", "sem descritores tímbricos completos", "evidência associativa indisponível"],
    ["Ritmo", "5 proxies de eventos", "5 proxies de eventos", "sem tempo, métrica, groove ou síncope", "baixo/moderado em arousal"],
    ["Textura", "ausente", "ausente", "sem camadas ou separação de fontes", "limita sobretudo valence"],
    ["Forma", "ausente", "ausente", "sem seções ou recorrência estrutural", "limita sobretudo valence"],
    ["Técnicas expressivas", "ausente", "ausente", "sem articulação ou performance", "limita valence e nuances"],
], columns=[
    "Categoria Panda", "Fingerprint Rank", "Fingerprint Band Rank",
    "Limitação", "Impacto esperado",
])
gap_map.to_csv(OUTPUT_DIR / "05_mapa_lacunas.csv", index=False, encoding="utf-8-sig")

fig_5 = go.Figure(data=[go.Table(
    header=dict(
        values=[f"<b>{c}</b>" for c in gap_map.columns],
        fill_color="#244a64",
        font=dict(color="white", size=12),
        align="left",
        height=34,
    ),
    cells=dict(
        values=[gap_map[c] for c in gap_map.columns],
        fill_color=[["#f4f8fb", "#ffffff"] * 4],
        align="left",
        font=dict(size=11),
        height=40,
    ),
    columnwidth=[105, 165, 190, 225, 155],
)])
fig_5.update_layout(
    title="Figura 5 — Mapa de lacunas da representação de fingerprinting",
    height=590,
    margin=dict(l=20, r=20, t=75, b=20),
)
save_plotly_figure(fig_5, "fig_05_mapa_lacunas")
fig_5.show()


### 11.1 Modo de análise de uma música

Quando ativado, este modo produz uma leitura temporal e ilustrativa da música selecionada. Ele não
deve ser usado para generalizar resultados do corpus.


In [13]:
if ANALISAR_APENAS_UMA_MUSICA:
    time_col = "block_start_sec" if "block_start_sec" in df_band.columns else "block_id"
    timeline = df_band.sort_values(["block_id"]).copy()

    fig_song_va = px.line(
        timeline, x=time_col, y=["valence", "arousal"], markers=True,
        title=f"Música {SONG_ID_ANALISE} — evolução temporal de valence e arousal",
        labels={"value": "Valor", "variable": "Target", time_col: "Tempo inicial do bloco (s)"},
        color_discrete_map=TARGET_COLORS,
    )
    improve_layout(fig_song_va)
    save_plotly_figure(fig_song_va, "musica_01_timeline_valence_arousal")
    fig_song_va.show()

    fig_song_energy = px.line(
        timeline, x=time_col, y=energy_norm, markers=True,
        title=f"Música {SONG_ID_ANALISE} — evolução de energy_norm por banda",
        labels={"value": "Energia normalizada", "variable": "Descritor", time_col: "Tempo inicial do bloco (s)"},
    )
    improve_layout(fig_song_energy)
    save_plotly_figure(fig_song_energy, "musica_02_energy_norm")
    fig_song_energy.show()

    fig_song_score = px.line(
        timeline, x=time_col, y=score_norm, markers=True,
        title=f"Música {SONG_ID_ANALISE} — evolução da saliência relativa por banda",
        labels={"value": "Score relativo", "variable": "Banda", time_col: "Tempo inicial do bloco (s)"},
    )
    improve_layout(fig_song_score)
    save_plotly_figure(fig_song_score, "musica_03_scores_bandas")
    fig_song_score.show()

    band_counts = (
        df_band_raw.groupby(["block_id", "band_name"], observed=True)
        .size().reset_index(name="n_picos")
    )
    fig_song_bands = px.bar(
        band_counts, x="block_id", y="n_picos", color="band_name", barmode="stack",
        category_orders={"band_name": BAND_ORDER},
        title=f"Música {SONG_ID_ANALISE} — distribuição dos picos por banda e bloco",
        labels={"block_id": "Bloco", "n_picos": "Quantidade de picos", "band_name": "Banda"},
    )
    improve_layout(fig_song_bands)
    save_plotly_figure(fig_song_bands, "musica_04_distribuicao_bandas")
    fig_song_bands.show()

    peak_size_col = "magnitude_norm" if "magnitude_norm" in df_band_raw.columns else None
    fig_song_peaks = px.scatter(
        df_band_raw,
        x="block_id",
        y="frequency_hz",
        color="band_name",
        size=peak_size_col,
        hover_data=["peak_rank_in_band", "magnitude_db"],
        category_orders={"band_name": BAND_ORDER},
        log_y=True,
        title=f"Música {SONG_ID_ANALISE} — top picos por bloco",
        labels={"block_id": "Bloco", "frequency_hz": "Frequência (Hz)", "band_name": "Banda"},
    )
    improve_layout(fig_song_peaks, height=540)
    save_plotly_figure(fig_song_peaks, "musica_05_top_picos")
    fig_song_peaks.show()
else:
    print("Modo de música única inativo; gráficos ilustrativos não foram gerados.")


Modo de música única inativo; gráficos ilustrativos não foram gerados.


## 12. Síntese textual pronta para o TCC

Os textos abaixo são gerados a partir das tabelas desta execução. No modo individual, recebem um aviso
explícito de que constituem apenas uma ilustração.


In [14]:
def category_value(scope, level, target, category="Dinâmica", technique="Combinado"):
    row = category_correlations[
        category_correlations["escopo"].eq(scope)
        & category_correlations["nivel_analise"].eq(level)
        & category_correlations["target"].eq(target)
        & category_correlations["categoria_panda"].eq(category)
        & category_correlations["tecnica"].eq(technique)
    ]
    return float(row["corr_abs_media"].iloc[0]) if len(row) and row["corr_abs_media"].notna().iloc[0] else np.nan

main_a = category_value("Principal (invariante a ganho)", "Bloco", "arousal")
main_v = category_value("Principal (invariante a ganho)", "Bloco", "valence")
sensitivity_a = category_value("Sensibilidade (todas as features)", "Bloco", "arousal")
sensitivity_v = category_value("Sensibilidade (todas as features)", "Bloco", "valence")
song_a = category_value("Principal (invariante a ganho)", "Música", "arousal")
song_v = category_value("Principal (invariante a ganho)", "Música", "valence")
rank_a = category_value(
    "Principal (invariante a ganho)", "Bloco", "arousal", technique="Rank"
)
rank_v = category_value(
    "Principal (invariante a ganho)", "Bloco", "valence", technique="Rank"
)
band_a = category_value(
    "Principal (invariante a ganho)", "Bloco", "arousal", technique="Band Rank"
)
band_v = category_value(
    "Principal (invariante a ganho)", "Bloco", "valence", technique="Band Rank"
)

prefix = (
    f"**Evidência ilustrativa da música {SONG_ID_ANALISE}; não generalizar para o corpus.**\n\n"
    if ANALISAR_APENAS_UMA_MUSICA else ""
)
if ANALISAR_APENAS_UMA_MUSICA:
    chapter_4 = (
        f"{prefix}A inspeção dos blocos disponíveis mostra como energia, saliência e distribuição "
        "espectral variam ao longo de um exemplo concreto. Como há apenas uma música, correlações "
        "agregadas e comparações entre quadrantes podem permanecer indisponíveis."
    )
    conclusion = (
        f"{prefix}O exemplo ilustra o tipo de evidência capturada pelo fingerprint, mas não permite "
        "inferir sozinho uma diferença geral entre arousal e valence."
    )
else:
    chapter_4 = (
        "A análise das 554 features de fingerprinting segundo a taxonomia de Panda et al. mostrou "
        "que a representação cobre principalmente dinâmica, saliência espectral, altura local e "
        "proxies fracos de harmonia e ritmo. Textura musical, forma e técnicas expressivas permanecem "
        f"ausentes. Na leitura principal invariante a ganho, Dinâmica apresentou correlação absoluta "
        f"média de {main_a:.3f} com arousal e {main_v:.3f} com valence por bloco. A agregação por "
        f"música preservou a assimetria ({song_a:.3f} contra {song_v:.3f}). Separando as técnicas, "
        f"Rank apresentou {rank_a:.3f}/{rank_v:.3f} e Band Rank apresentou "
        f"{band_a:.3f}/{band_v:.3f} para arousal/valence, respectivamente."
    )
    conclusion = (
        "Os resultados indicam que a contribuição dos fingerprints é seletiva. A representação é "
        "mais adequada para capturar variações de arousal, pois descreve energia, brilho, magnitude "
        "relativa e saliência por banda. A predição de valence permanece limitada porque depende de "
        "fatores musicais de maior nível, como modo, progressões harmônicas, textura, forma e "
        "expressividade. A análise de sensibilidade com medidas absolutas mantém a mesma direção "
        f"({sensitivity_a:.3f} para arousal e {sensitivity_v:.3f} para valence), mas pode incorporar "
        "efeitos de ganho, mixagem e masterização."
    )

limitations = (
    prefix
    + "As categorias musicais inferidas a partir dos nomes das features devem ser interpretadas como "
    "proxies acústicos. O método não realiza transcrição musical, identificação de acordes, extração "
    "de forma musical, separação de textura ou análise expressiva completa. Correlações são "
    "descritivas e não substituem validação preditiva."
)
texts = (
    "# Texto para o Capítulo 4 — Resultados interpretativos\n\n"
    + chapter_4
    + "\n\n# Texto para o Capítulo 5 — Conclusão\n\n"
    + conclusion
    + "\n\n# Texto para Limitações\n\n"
    + limitations
)
(OUTPUT_DIR / "06_textos_prontos_tcc.md").write_text(texts, encoding="utf-8")
display(Markdown(texts))


# Texto para o Capítulo 4 — Resultados interpretativos

A análise das 554 features de fingerprinting segundo a taxonomia de Panda et al. mostrou que a representação cobre principalmente dinâmica, saliência espectral, altura local e proxies fracos de harmonia e ritmo. Textura musical, forma e técnicas expressivas permanecem ausentes. Na leitura principal invariante a ganho, Dinâmica apresentou correlação absoluta média de 0.281 com arousal e 0.072 com valence por bloco. A agregação por música preservou a assimetria (0.295 contra 0.136). Separando as técnicas, Rank apresentou 0.249/0.063 e Band Rank apresentou 0.298/0.077 para arousal/valence, respectivamente.

# Texto para o Capítulo 5 — Conclusão

Os resultados indicam que a contribuição dos fingerprints é seletiva. A representação é mais adequada para capturar variações de arousal, pois descreve energia, brilho, magnitude relativa e saliência por banda. A predição de valence permanece limitada porque depende de fatores musicais de maior nível, como modo, progressões harmônicas, textura, forma e expressividade. A análise de sensibilidade com medidas absolutas mantém a mesma direção (0.347 para arousal e 0.112 para valence), mas pode incorporar efeitos de ganho, mixagem e masterização.

# Texto para Limitações

As categorias musicais inferidas a partir dos nomes das features devem ser interpretadas como proxies acústicos. O método não realiza transcrição musical, identificação de acordes, extração de forma musical, separação de textura ou análise expressiva completa. Correlações são descritivas e não substituem validação preditiva.

## 13. Apêndice técnico opcional

O apêndice é executado somente quando `EXECUTAR_APENDICE_TECNICO = True`. Ele reúne auditoria dos raw
peaks, erro em cents, harmonicidade aproximada e entropia espectral. Essas métricas aprofundam a
rastreabilidade acústica, mas não fazem parte da evidência principal e não são adicionadas aos
Parquets ou aos experimentos de modelagem.


In [15]:
if EXECUTAR_APENDICE_TECNICO:
    appendix_dir = OUTPUT_DIR / "apendice_tecnico"
    appendix_dir.mkdir(parents=True, exist_ok=True)

    raw_audit = pd.DataFrame([
        {
            "origem": "Rank",
            "linhas": len(df_rank_raw),
            "musicas": df_rank_raw["song_id"].nunique(),
            "blocos": df_rank_raw[["song_id", "block_id"]].drop_duplicates().shape[0],
            "picos_por_bloco": len(df_rank_raw) / max(
                1, df_rank_raw[["song_id", "block_id"]].drop_duplicates().shape[0]
            ),
        },
        {
            "origem": "Band Rank",
            "linhas": len(df_band_raw),
            "musicas": df_band_raw["song_id"].nunique(),
            "blocos": df_band_raw[["song_id", "block_id"]].drop_duplicates().shape[0],
            "picos_por_bloco": len(df_band_raw) / max(
                1, df_band_raw[["song_id", "block_id"]].drop_duplicates().shape[0]
            ),
        },
    ])
    raw_audit.to_csv(appendix_dir / "A01_auditoria_raw_peaks.csv", index=False, encoding="utf-8-sig")
    display(raw_audit)

    raw_frames = []
    for origin, raw in [("Rank", df_rank_raw), ("Band Rank", df_band_raw)]:
        temp = raw.copy()
        temp["origem"] = origin
        freq = pd.to_numeric(temp["frequency_hz"], errors="coerce")
        valid = freq > 0
        midi_float = pd.Series(np.nan, index=temp.index, dtype=float)
        midi_float.loc[valid] = 69 + 12 * np.log2(freq.loc[valid] / 440.0)
        nearest = midi_float.round()
        nearest_hz = pd.Series(np.nan, index=temp.index, dtype=float)
        nearest_hz.loc[valid] = 440.0 * 2 ** ((nearest.loc[valid] - 69) / 12)
        temp["erro_cents_abs"] = abs(1200 * np.log2(freq / nearest_hz))
        raw_frames.append(temp)
    raw_all = pd.concat(raw_frames, ignore_index=True)

    cents_summary = (
        raw_all.groupby("origem")
        .agg(
            n_picos=("erro_cents_abs", "count"),
            erro_cents_medio=("erro_cents_abs", "mean"),
            erro_cents_mediano=("erro_cents_abs", "median"),
            pct_ate_25_cents=("erro_cents_abs", lambda s: (s <= 25).mean() * 100),
            pct_ate_50_cents=("erro_cents_abs", lambda s: (s <= 50).mean() * 100),
        )
        .reset_index()
    )
    cents_summary.to_csv(
        appendix_dir / "A02_erro_cents.csv", index=False, encoding="utf-8-sig"
    )
    cents_summary.to_latex(
        appendix_dir / "A02_erro_cents.tex", index=False, float_format="%.3f"
    )
    display(cents_summary)

    def harmonicity_for_group(group):
        freq = np.sort(
            pd.to_numeric(group["frequency_hz"], errors="coerce")
            .replace([np.inf, -np.inf], np.nan).dropna().to_numpy()
        )
        freq = freq[freq > 0]
        if len(freq) < 2:
            return pd.Series({
                "n_picos": len(freq),
                "harmonic_like_ratio": np.nan,
                "erro_harmonico_cents_medio": np.nan,
            })
        fundamental = freq[0]
        ratios = freq / fundamental
        nearest_harmonic = np.maximum(1, np.round(ratios))
        errors = abs(1200 * np.log2(ratios / nearest_harmonic))
        return pd.Series({
            "n_picos": len(freq),
            "harmonic_like_ratio": float(np.mean(errors <= 35)),
            "erro_harmonico_cents_medio": float(np.mean(errors)),
        })

    harmonic_frames = []
    for origin, raw in [("Rank", df_rank_raw), ("Band Rank", df_band_raw)]:
        result = (
            raw.groupby(["song_id", "block_id"], observed=True)
            .apply(harmonicity_for_group, include_groups=False)
            .reset_index()
        )
        result["origem"] = origin
        harmonic_frames.append(result)
    harmonicity = pd.concat(harmonic_frames, ignore_index=True)
    harmonicity.to_csv(
        appendix_dir / "A03_harmonicidade_aproximada.csv",
        index=False, encoding="utf-8-sig",
    )
    harmonicity_summary = (
        harmonicity.groupby("origem", as_index=False)
        .agg(
            blocos=("block_id", "size"),
            harmonic_like_ratio_medio=("harmonic_like_ratio", "mean"),
            erro_harmonico_cents_medio=("erro_harmonico_cents_medio", "mean"),
        )
    )
    display(harmonicity_summary)

    def normalized_entropy(values):
        counts = pd.Series(values).dropna().value_counts().to_numpy(dtype=float)
        if len(counts) <= 1 or counts.sum() == 0:
            return 0.0
        probabilities = counts / counts.sum()
        entropy = -np.sum(probabilities * np.log2(probabilities))
        return float(entropy / np.log2(len(counts)))

    entropy_frames = []
    for origin, raw in [("Rank", df_rank_raw), ("Band Rank", df_band_raw)]:
        band_col = "freq_band" if "freq_band" in raw.columns else "band_name"
        result = (
            raw.groupby(["song_id", "block_id"], observed=True)
            .agg(
                entropia_banda=(band_col, normalized_entropy),
                entropia_oitava=("octave", normalized_entropy),
                entropia_pitch_class=("pitch_class", normalized_entropy),
            )
            .reset_index()
        )
        result["origem"] = origin
        entropy_frames.append(result)
    entropy_blocks = pd.concat(entropy_frames, ignore_index=True)
    entropy_blocks.to_csv(
        appendix_dir / "A04_entropia_espectral.csv",
        index=False, encoding="utf-8-sig",
    )
    entropy_summary = (
        entropy_blocks.groupby("origem", as_index=False)
        .agg(
            entropia_banda_media=("entropia_banda", "mean"),
            entropia_oitava_media=("entropia_oitava", "mean"),
            entropia_pitch_class_media=("entropia_pitch_class", "mean"),
        )
    )
    entropy_summary.to_latex(
        appendix_dir / "A04_entropia_espectral.tex",
        index=False, float_format="%.4f",
    )
    display(entropy_summary)
else:
    print("Apêndice técnico não executado. Ative EXECUTAR_APENDICE_TECNICO para recalculá-lo.")


Apêndice técnico não executado. Ative EXECUTAR_APENDICE_TECNICO para recalculá-lo.


### Verificação final de integridade e aceitação

Esta célula confirma as contagens esperadas, a direção do achado principal, a geração das tabelas e
figuras e a preservação dos timestamps dos Parquets.


In [16]:
INPUT_PARQUET_MTIMES_AFTER = {name: path.stat().st_mtime_ns for name, path in PATHS.items()}
changed_inputs = {
    name: (INPUT_PARQUET_MTIMES_BEFORE[name], INPUT_PARQUET_MTIMES_AFTER[name])
    for name in PATHS
    if INPUT_PARQUET_MTIMES_BEFORE[name] != INPUT_PARQUET_MTIMES_AFTER[name]
}
assert not changed_inputs, f"Entradas foram alteradas: {changed_inputs}"
assert len(df_features) == 554, f"Esperadas 554 features; obtidas {len(df_features)}"
assert invariant_count == 467, f"Esperadas 467 features invariantes; obtidas {invariant_count}"
assert absolute_dynamic_count == 87, f"Esperadas 87 features absolutas; obtidas {absolute_dynamic_count}"
assert nao_classificadas.empty, "Há features não classificadas."

if ANALISAR_APENAS_UMA_MUSICA:
    assert len(df_rank) == 3, f"Esperados 3 blocos Rank; obtidos {len(df_rank)}"
    assert len(df_band) == 3, f"Esperados 3 blocos Band Rank; obtidos {len(df_band)}"
    assert len(df_rank_raw) == 90, f"Esperados 90 raw peaks Rank; obtidos {len(df_rank_raw)}"
    assert len(df_band_raw) == 120, f"Esperados 120 raw peaks Band Rank; obtidos {len(df_band_raw)}"
else:
    assert len(df_rank) == 6506 and len(df_band) == 6506
    assert df_rank["song_id"].nunique() == 1802
    checks = [
        ("Principal (invariante a ganho)", "Bloco"),
        ("Principal (invariante a ganho)", "Música"),
        ("Sensibilidade (todas as features)", "Bloco"),
        ("Sensibilidade (todas as features)", "Música"),
    ]
    for scope_name, level in checks:
        arousal_value = category_value(scope_name, level, "arousal")
        valence_value = category_value(scope_name, level, "valence")
        assert arousal_value > valence_value, (
            f"Direção inesperada em {scope_name}/{level}: "
            f"arousal={arousal_value}, valence={valence_value}"
        )
    assert abs(main_a - 0.2808) < 0.01, main_a
    assert abs(main_v - 0.0719) < 0.01, main_v
    assert abs(sensitivity_a - 0.3471) < 0.01, sensitivity_a
    assert abs(sensitivity_v - 0.1118) < 0.01, sensitivity_v
    for technique in ["Rank", "Band Rank"]:
        technique_a = category_value(
            "Principal (invariante a ganho)", "Bloco", "arousal",
            technique=technique,
        )
        technique_v = category_value(
            "Principal (invariante a ganho)", "Bloco", "valence",
            technique=technique,
        )
        assert technique_a > technique_v, (
            f"Direção inesperada para {technique}: "
            f"arousal={technique_a}, valence={technique_v}"
        )

required_tables = [
    "01_tabela_cobertura_panda.csv",
    "01_tabela_cobertura_rank.csv",
    "01_tabela_cobertura_band_rank.csv",
    "02_tabela_correlacao_categoria.csv",
    "02_tabela_correlacao_rank.csv",
    "02_tabela_correlacao_band_rank.csv",
    "03_tabela_diferenca_arousal_valence.csv",
    "03_tabela_diferenca_rank.csv",
    "03_tabela_diferenca_band_rank.csv",
    "04_tabela_perfil_quadrante.csv",
]
required_figures = [
    "fig_01_cobertura_panda.html",
    "fig_02_correlacao_categoria.html",
    "fig_03_razao_arousal_valence.html",
    "fig_04_heatmap_quadrantes.html",
    "fig_05_mapa_lacunas.html",
]
missing_outputs = [
    name for name in required_tables + required_figures
    if not (OUTPUT_DIR / name).exists()
]
assert not missing_outputs, f"Saídas obrigatórias ausentes: {missing_outputs}"

if EXECUTAR_APENDICE_TECNICO:
    appendix_expected = [
        "A01_auditoria_raw_peaks.csv",
        "A02_erro_cents.csv",
        "A03_harmonicidade_aproximada.csv",
        "A04_entropia_espectral.csv",
    ]
    appendix_missing = [
        name for name in appendix_expected
        if not (OUTPUT_DIR / "apendice_tecnico" / name).exists()
    ]
    assert not appendix_missing, f"Artefatos do apêndice ausentes: {appendix_missing}"

validation = {
    "modo": RUN_NAME,
    "blocos_rank": len(df_rank),
    "blocos_band_rank": len(df_band),
    "musicas": int(df_rank["song_id"].nunique()),
    "features_total": len(df_features),
    "features_invariantes_ganho": invariant_count,
    "features_absolutas_sensibilidade": absolute_dynamic_count,
    "features_nao_classificadas": len(nao_classificadas),
    "dinamica_principal_bloco_arousal": None if np.isnan(main_a) else main_a,
    "dinamica_principal_bloco_valence": None if np.isnan(main_v) else main_v,
    "dinamica_sensibilidade_bloco_arousal": None if np.isnan(sensitivity_a) else sensitivity_a,
    "dinamica_sensibilidade_bloco_valence": None if np.isnan(sensitivity_v) else sensitivity_v,
    "rank_dinamica_principal_arousal": None if np.isnan(rank_a) else rank_a,
    "rank_dinamica_principal_valence": None if np.isnan(rank_v) else rank_v,
    "band_rank_dinamica_principal_arousal": None if np.isnan(band_a) else band_a,
    "band_rank_dinamica_principal_valence": None if np.isnan(band_v) else band_v,
    "parquets_preservados": not changed_inputs,
    "apendice_executado": EXECUTAR_APENDICE_TECNICO,
}
(OUTPUT_DIR / "validacao_execucao.json").write_text(
    json.dumps(validation, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
display(pd.DataFrame([validation]))
print("Validação concluída sem alterar os Parquets originais.")


,modo,blocos_rank,blocos_band_rank,musicas,features_total,features_invariantes_ganho,features_absolutas_sensibilidade,features_nao_classificadas,dinamica_principal_bloco_arousal,dinamica_principal_bloco_valence,dinamica_sensibilidade_bloco_arousal,dinamica_sensibilidade_bloco_valence,rank_dinamica_principal_arousal,rank_dinamica_principal_valence,band_rank_dinamica_principal_arousal,band_rank_dinamica_principal_valence,parquets_preservados,apendice_executado
0,corpus,6506,6506,1802,554,467,87,0,0.280759,0.071941,0.347064,0.111785,0.248528,0.062701,0.297985,0.07688,True,False


Validação concluída sem alterar os Parquets originais.
